# MPB LaTeX OCR - Colab GPU Training

This notebook runs the project on a Colab GPU. Use `Runtime > Change runtime type > GPU` before running the cells.

The default cells train the smoke-test model on toy data. Replace the manifest and image-root paths when you move to Im2LaTeX or UniMER data.

In [ ]:
import os
import subprocess
from pathlib import Path

def run(command, cwd=None):
    print(f"$ {command}")
    subprocess.run(command, shell=True, check=True, cwd=cwd)

run("nvidia-smi")

## Get The Project

If this notebook is already running from a cloned copy of the repo, set `PROJECT_DIR` to that folder. Otherwise set `REPO_URL` to your Git repository URL and run the clone cell.

In [ ]:
REPO_URL = ""  # Example: "https://github.com/YOUR_USER/mpb-latex-ocr.git"
PROJECT_DIR = Path("/content/mpb-latex-ocr")

if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL, or upload/clone the repo to /content/mpb-latex-ocr first.")
    run(f"git clone {REPO_URL} {PROJECT_DIR}")

print(PROJECT_DIR)

## Optional: Persist Runs To Google Drive

Colab's `/content` storage disappears when the runtime resets. Enable this cell if you want checkpoints and MLflow runs saved to Drive.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/mpb-latex-ocr")
else:
    RUN_ROOT = Path("/content")

OUTPUT_DIR = RUN_ROOT / "latex-ocr-runs" / "baseline"
MLFLOW_DIR = RUN_ROOT / "mlruns"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MLFLOW_DIR:", MLFLOW_DIR)

## Install Dependencies

Colab normally already has a CUDA-enabled PyTorch build. This installs the project package and its missing training dependencies.

In [ ]:
run("python -m pip install --upgrade pip", cwd=PROJECT_DIR)
run('python -m pip install -e ".[dev]"', cwd=PROJECT_DIR)

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Generate Toy Data

This is only for smoke testing. Real training should use a manifest for Im2LaTeX, UniMER, CROHME, or MathWriting.

In [ ]:
run("latex-ocr-make-toy-data --output-dir data/toy --num-samples 240", cwd=PROJECT_DIR)

## Train

The config uses Colab-safe defaults: one GPU, `16-mixed`, batch size 16, and gradient accumulation.

In [ ]:
train_cmd = " ".join([
    "latex-ocr-train",
    "--config configs/train.yaml",
    "--config configs/hardware/colab_gpu.yaml",
    "trainer.max_epochs=3",
    f"paths.output_dir={OUTPUT_DIR}",
    f"paths.tokenizer_path={OUTPUT_DIR / 'tokenizer.json'}",
    f"mlflow.tracking_uri=file:{MLFLOW_DIR}",
])
run(train_cmd, cwd=PROJECT_DIR)

## Evaluate The Best Checkpoint

In [ ]:
checkpoints = sorted((OUTPUT_DIR / "checkpoints").glob("*.ckpt"))
if not checkpoints:
    raise FileNotFoundError(f"No checkpoints found under {OUTPUT_DIR / 'checkpoints'}")
checkpoint = checkpoints[-1]
print("checkpoint:", checkpoint)

eval_cmd = " ".join([
    "latex-ocr-evaluate",
    f"--checkpoint {checkpoint}",
    "--manifest data/toy/manifest.csv",
    "--image-root data/toy",
    f"--tokenizer {OUTPUT_DIR / 'tokenizer.json'}",
    "--split test",
    f"--predictions-out {OUTPUT_DIR / 'test_predictions.jsonl'}",
])
run(eval_cmd, cwd=PROJECT_DIR)

## View MLflow Runs

Colab cannot expose the local MLflow UI as cleanly as a normal machine. The run files are saved under `MLFLOW_DIR`; download them or inspect them later from a local MLflow UI.